# AnimalCLEF association and discovery metrics

Apply the selected parameters to their WildFusion matrices. Report ARI, AMI, V-measure, item-averaged B³ and pairwise F1 for all queries and novel queries. Compute attachment, BAKS, BAUS and NormAcc for the gallery settings. Set APEXID_DATASET_NAME for a single-dataset run.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("APEXID_REPO_ROOT", Path.cwd())).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "apexid").is_dir():
    raise FileNotFoundError("Start the notebook in the repository or set APEXID_REPO_ROOT.")
sys.path.insert(0, str(REPO_ROOT))


from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
    v_measure_score,
)


GLOBAL_DESCRIPTOR = os.environ["APEXID_GLOBAL_DESCRIPTOR"]
ALLOWED_DATASETS = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")
DATASET_NAME = os.environ.get("APEXID_DATASET_NAME")
if DATASET_NAME is not None and DATASET_NAME not in ALLOWED_DATASETS:
    raise ValueError(f"Choose a dataset from {ALLOWED_DATASETS}")
SELECTED_DATASETS = (DATASET_NAME,) if DATASET_NAME else ALLOWED_DATASETS
SPLIT_SEEDS = (42,)
stage03 = REPO_ROOT / "outputs" / "stage03"
FROZEN_CONFIGS = {}


def canonical_global_descriptor(name):
    return name.replace("ApexID_MiewID_", "ApexID_Miewid_", 1) if name.startswith("ApexID_MiewID_") else name

def protocol_for(dataset_name):
    return "query_only_defaults" if dataset_name == "jaguar_antonio" else "gallery_attach_optuna"

def find_artifact(filename, explicit_env, dataset_name=None):
    explicit = os.environ.get(explicit_env)
    if explicit:
        path = Path(explicit)
        if path.name != filename or not path.is_file():
            raise FileNotFoundError(f"{explicit_env} must point to {filename}")
        return path
    has_matrix = filename.startswith("sim_matrix_")
    variable, subdir = ("APEXID_STAGE03A_OUTPUT", "03a") if has_matrix else ("APEXID_STAGE03B_OUTPUT", "03b")
    root = Path(os.environ.get(variable, stage03 / subdir)) / GLOBAL_DESCRIPTOR
    for path in (root / (dataset_name or "") / filename, root / filename):
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f"Expected {filename} under {root}.")

from apexid.evaluation.animalclef_open_set import (
    chinese_whispers,
    compute_identity_scores,
    re_ranking,
    resolve_cluster_labels,
    summarize_best_identities,
)

local_out = stage03 / "03c" / GLOBAL_DESCRIPTOR
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03C_OUTPUT",
    str(local_out),
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

jobs = []
missing = []
for dataset_name in SELECTED_DATASETS:
    matrix_name = f"sim_matrix_{GLOBAL_DESCRIPTOR}_{dataset_name}_open.npy"
    config_name = f"best_configs_{GLOBAL_DESCRIPTOR}_{dataset_name}.json"
    try:
        matrix_path = find_artifact(matrix_name, "APEXID_STAGE03A_MATRIX", dataset_name)
    except FileNotFoundError as exc:
        missing.append(f"{dataset_name}: {exc}")
        continue
    config_path = None
    try:
        config_path = find_artifact(config_name, "APEXID_STAGE03B_CONFIG", dataset_name)
        config_payload = json.loads(config_path.read_text(encoding="utf-8"))
        if dataset_name not in config_payload:
            raise KeyError(f"{config_name} lacks key {dataset_name}")
        config = config_payload[dataset_name]
    except (FileNotFoundError, KeyError):
        if dataset_name not in FROZEN_CONFIGS:
            missing.append(f"{dataset_name}: no config JSON and no baked FROZEN_CONFIGS entry")
            continue
        config = dict(FROZEN_CONFIGS[dataset_name])
        config_path = OUTPUT_ROOT / config_name
        config_path.write_text(json.dumps({dataset_name: config}, indent=2, sort_keys=True), encoding="utf-8")
        print(f"{dataset_name}: using baked FROZEN_CONFIGS -> wrote {config_path}")
    else:
        # Prefer explicit Optuna JSON when present; still allow jaguar baked defaults.
        config = config_payload[dataset_name]
    jobs.append({
        "dataset": dataset_name,
        "protocol": str(config.get("protocol") or protocol_for(dataset_name)),
        "matrix_path": matrix_path,
        "config_path": config_path,
        "config": config,
    })

if missing:
    raise FileNotFoundError(
        "03c needs each animal's 03a open matrix (qq+qd inside one file). Missing:\n- "
        + "\n- ".join(missing)
    )
print("Descriptor:", GLOBAL_DESCRIPTOR)
print("GLOBAL_DESCRIPTOR:", GLOBAL_DESCRIPTOR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SELECTED_DATASETS:", [job["dataset"] for job in jobs])
for job in jobs:
    print(
        f"{job['dataset']}: matrix={job['matrix_path']} "
        f"config={job['config_path']} protocol={job['protocol']}"
    )

In [ ]:
# Load each animal's qq/qd matrices with that animal's frozen 03b thresholds.
def load_animal(job):
    dataset_name = job["dataset"]
    protocol = job["protocol"]
    obj = np.load(job["matrix_path"], allow_pickle=True).item()
    if dataset_name not in obj:
        raise KeyError(f"{job['matrix_path']} keys {list(obj)} do not include {dataset_name}")
    bundle = obj[dataset_name]
    sim_qq = np.asarray(bundle["sim_query_query"], dtype=np.float32)
    sim_qg = np.asarray(bundle.get("sim_query_db", np.zeros((len(sim_qq), 0), dtype=np.float32)), dtype=np.float32)
    query_image_ids = np.asarray(bundle["query_image_ids"]).astype(str)
    db_image_ids = np.asarray(bundle["db_image_ids"]).astype(str)
    y_query = np.asarray(bundle["query_identities"]).astype(str)
    y_db = np.asarray(bundle["db_identities"]).astype(str)
    query_open_role = np.asarray(bundle["query_open_role"]).astype(str)
    if canonical_global_descriptor(str(bundle.get("global_descriptor", ""))) != GLOBAL_DESCRIPTOR:
        raise ValueError(f"{dataset_name}: 03a global descriptor does not match the 03c knob")
    if bundle.get("dataset") not in {None, dataset_name}:
        raise ValueError(f"{dataset_name}: 03a dataset tag mismatch")
    if sim_qq.shape != (len(query_image_ids), len(query_image_ids)):
        raise ValueError(f"{dataset_name}: query-query shape does not match query_image_ids")
    if sim_qg.shape != (len(query_image_ids), len(db_image_ids)):
        raise ValueError(f"{dataset_name}: query-gallery shape does not match ids")
    if len(set(query_image_ids)) != len(query_image_ids):
        raise ValueError(f"{dataset_name}: query image_ids are not unique")

    if "config" in job and job["config"] is not None:
        config = dict(job["config"])
    else:
        config_payload = json.loads(job["config_path"].read_text(encoding="utf-8"))
        if dataset_name not in config_payload:
            raise KeyError(f"{job['config_path']} does not contain {dataset_name}")
        config = config_payload[dataset_name]
    cfg_desc = canonical_global_descriptor(str(config.get("global_descriptor", "")))
    if config.get("dataset") != dataset_name or cfg_desc != GLOBAL_DESCRIPTOR:
        raise ValueError(f"{dataset_name}: frozen config dataset/model does not match")
    if config.get("protocol") != protocol:
        raise ValueError(f"{dataset_name}: frozen config protocol {config.get('protocol')!r} != {protocol!r}")
    required_config = {
        "cw_threshold", "cw_top_k", "cw_iterations", "cw_k_core_degree",
        "use_rerank", "rr_k1", "rr_k2", "rr_lambda",
    }
    if protocol == "gallery_attach_optuna":
        required_config.update({"top_r", "tau_attach", "tau_margin"})
    missing_keys = required_config - set(config)
    if missing_keys:
        raise ValueError(f"{dataset_name}: frozen config is missing {sorted(missing_keys)}")
    if protocol == "query_only_defaults" and (
        config.get("database_required") is not False
        or config.get("known_attachment_enabled") is not False
        or int(config.get("n_trials_completed", -1)) != 0
    ):
        raise ValueError(f"{dataset_name}: query-only config must disable database, attachment, and tuning")
    return {
        "dataset": dataset_name,
        "protocol": protocol,
        "bundle": bundle,
        "config": config,
        "sim_qq": sim_qq,
        "sim_qg": sim_qg,
        "query_image_ids": query_image_ids,
        "db_image_ids": db_image_ids,
        "y_query": y_query,
        "y_db": y_db,
        "query_open_role": query_open_role,
        "matrix_path": job["matrix_path"],
        "config_path": job["config_path"],
    }


loaded_animals = [load_animal(job) for job in jobs]
for animal in loaded_animals:
    print(
        f"{animal['dataset']}: qq={animal['sim_qq'].shape} qd={animal['sim_qg'].shape} "
        f"protocol={animal['protocol']}"
    )
    print(json.dumps(animal["config"], indent=2, sort_keys=True))


In [ ]:
# Frozen evaluation: each animal uses only its own 03b thresholds.
def _balanced_mean(labels, values):
    """Identity-balanced mean used by Stage 02 BAKS/BAUS."""
    labels_array = np.asarray(labels).astype(str)
    values_array = np.asarray(values, dtype=float)
    if not len(values_array):
        return float("nan")
    return float(np.mean([values_array[labels_array == identity].mean() for identity in np.unique(labels_array)]))


from apexid.evaluation.benchmark_metrics import b3_metrics, pairwise_metrics


def clustering_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)
    b3 = b3_metrics(y_true, y_pred)
    b_p, b_r, b_f = b3["b3_precision"], b3["b3_recall"], b3["b3_f1"]
    pairs = pairwise_metrics(y_true, y_pred)
    p_p, p_r, p_f = pairs["pairwise_precision"], pairs["pairwise_recall"], pairs["pairwise_f1"]
    return {
        f"{prefix}ARI": float(adjusted_rand_score(y_true, y_pred)),
        f"{prefix}AMI": float(adjusted_mutual_info_score(y_true, y_pred)),
        f"{prefix}NMI": float(normalized_mutual_info_score(y_true, y_pred)),
        f"{prefix}V_measure": float(v_measure_score(y_true, y_pred)),
        f"{prefix}homogeneity": float(homogeneity_score(y_true, y_pred)),
        f"{prefix}completeness": float(completeness_score(y_true, y_pred)),
        f"{prefix}BCubed_precision": b_p,
        f"{prefix}BCubed_recall": b_r,
        f"{prefix}BCubed_F1": b_f,
        f"{prefix}pairwise_precision": p_p,
        f"{prefix}pairwise_recall": p_r,
        f"{prefix}pairwise_F1": p_f,
        f"{prefix}n_true_identities": int(pd.Series(y_true).nunique()),
        f"{prefix}n_pred_clusters": int(pd.Series(y_pred).nunique()),
    }


def attachment_open_set_metrics(y_true, y_pred, known_mask, novel_mask, gallery_identities):
    """Stage-02 BAKS/BAUS/NormAcc with 'accepted' = attached to a gallery identity."""
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)
    known_mask = np.asarray(known_mask, dtype=bool)
    novel_mask = np.asarray(novel_mask, dtype=bool)
    gallery = set(map(str, gallery_identities))
    accepted = np.asarray([label in gallery for label in y_pred], dtype=bool)
    if known_mask.any():
        known_correct = (y_pred[known_mask] == y_true[known_mask])
        baks = _balanced_mean(y_true[known_mask], (accepted[known_mask] & known_correct).astype(float))
        known_to_new = float(np.mean(~accepted[known_mask]))
    else:
        baks = float("nan")
        known_to_new = float("nan")
    if novel_mask.any():
        baus = _balanced_mean(y_true[novel_mask], (~accepted[novel_mask]).astype(float))
        novel_false_attach = float(np.mean(accepted[novel_mask]))
    else:
        baus = float("nan")
        novel_false_attach = float("nan")
    if np.isfinite(baks) and np.isfinite(baus):
        import math
        normacc = float(math.sqrt(max(baks, 0.0) * max(baus, 0.0)))
    else:
        normacc = float("nan")
    return {
        "BAKS": baks,
        "BAUS": baus,
        "NormAcc": normacc,
        "known_attachment_accuracy": float(np.mean(y_pred[known_mask] == y_true[known_mask])) if known_mask.any() else float("nan"),
        "known_to_new_rate": known_to_new,
        "novel_false_attach_rate": novel_false_attach,
    }


def cluster_query_matrix(sim_qq_matrix, config):
    use_rerank = bool(config["use_rerank"])
    sim_for_cw = re_ranking(
        sim_qq_matrix,
        k1=int(config["rr_k1"]),
        k2=int(config["rr_k2"]),
        lambda_value=float(config["rr_lambda"]),
    ) if use_rerank else sim_qq_matrix
    return chinese_whispers(
        sim_for_cw,
        threshold=float(config["cw_threshold"]),
        iterations=int(config["cw_iterations"]),
        top_k=int(config["cw_top_k"]),
        k_core_degree=int(config["cw_k_core_degree"]),
        seed=int(config.get("cw_seed", 42)),
        skip_mutual_knn=use_rerank,
    )


def evaluate_animal(animal):
    dataset_name = animal["dataset"]
    protocol = animal["protocol"]
    config = animal["config"]
    sim_qq = animal["sim_qq"]
    query_image_ids = animal["query_image_ids"]
    y_query = animal["y_query"]
    if protocol == "query_only_defaults":
        cluster_ids = cluster_query_matrix(sim_qq, config)
        predicted = np.asarray(
            [f"cluster_{dataset_name}_new_{int(cluster)}" for cluster in cluster_ids],
            dtype=str,
        )
        cluster_metrics = clustering_metrics(y_query, predicted)
        # All-novel pool: BAKS undefined; BAUS=1 if nothing is treated as a gallery ID.
        open_metrics = attachment_open_set_metrics(
            y_query, predicted,
            known_mask=np.zeros(len(y_query), dtype=bool),
            novel_mask=np.ones(len(y_query), dtype=bool),
            gallery_identities=[],
        )
        result = {
            "global_descriptor": GLOBAL_DESCRIPTOR,
            "dataset": dataset_name,
            "protocol": protocol,
            "split_seed": "full_pool",
            **cluster_metrics,
            "novel_only_ARI": cluster_metrics["ARI"],
            "novel_only_AMI": cluster_metrics["AMI"],
            "novel_only_V_measure": cluster_metrics["V_measure"],
            "novel_only_BCubed_F1": cluster_metrics["BCubed_F1"],
            **open_metrics,
            "queries": len(query_image_ids),
            "gallery_images": 0,
            "query_known_images": 0,
            "query_novel_images": len(query_image_ids),
            "novel_identities": cluster_metrics["n_true_identities"],
            "predicted_clusters": cluster_metrics["n_pred_clusters"],
            "config_source": config.get("config_source"),
            "cw_threshold": float(config["cw_threshold"]),
            "cw_top_k": int(config["cw_top_k"]),
            "tau_attach": float("nan"),
            "tau_margin": float("nan"),
        }
        n_q = len(query_image_ids)
        details = pd.DataFrame({
            "global_descriptor": [GLOBAL_DESCRIPTOR] * n_q,
            "dataset": [dataset_name] * n_q,
            "protocol": [protocol] * n_q,
            "split_seed": ["full_pool"] * n_q,
            "image_id": query_image_ids,
            "open_role": ["query_novel"] * n_q,
            "true_identity": y_query,
            "raw_cluster_id": cluster_ids,
            "predicted_label": predicted,
            "best_known_identity": [""] * n_q,
            "best_known_score": np.full(n_q, np.nan, dtype=np.float64),
            "second_best_score": np.full(n_q, np.nan, dtype=np.float64),
            "known_margin": np.full(n_q, np.nan, dtype=np.float64),
        })
        print(
            f"{dataset_name} full_pool: ARI={result['ARI']:.4f} AMI={result['AMI']:.4f} "
            f"V={result['V_measure']:.4f} BCubedF1={result['BCubed_F1']:.4f} "
            f"BAUS={result['BAUS']:.4f} clusters={result['predicted_clusters']}"
        )
        return result, details

    query_open_role = animal["query_open_role"]
    db_image_ids = animal["db_image_ids"]
    y_db = animal["y_db"]
    sim_qg = animal["sim_qg"]
    known_mask = query_open_role == "query_known"
    novel_mask = query_open_role == "query_novel"
    if not known_mask.any() or not novel_mask.any():
        raise RuntimeError(f"{dataset_name}: open split must contain known and novel queries")
    if set(query_image_ids) & set(db_image_ids):
        raise RuntimeError(f"{dataset_name}: gallery/query overlap")
    novel_identities = set(y_query[novel_mask])
    if novel_identities & set(y_db):
        raise RuntimeError(f"{dataset_name}: novel identities occur in the gallery")

    identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r=int(config["top_r"]))
    best_names, best_scores, second_scores, margins = summarize_best_identities(identity_names, identity_scores)
    cluster_ids = cluster_query_matrix(sim_qq, config)
    predicted = resolve_cluster_labels(
        cluster_ids, best_names, best_scores, margins,
        tau_attach=float(config["tau_attach"]),
        tau_margin=float(config["tau_margin"]),
    )
    cluster_metrics = clustering_metrics(y_query, predicted)
    novel_metrics = clustering_metrics(y_query[novel_mask], predicted[novel_mask], prefix="novel_only_")
    open_metrics = attachment_open_set_metrics(y_query, predicted, known_mask, novel_mask, y_db)
    split_seed = animal["bundle"].get("split_seed") or (SPLIT_SEEDS[0] if SPLIT_SEEDS else 42)
    result = {
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "dataset": dataset_name,
        "protocol": protocol,
        "split_seed": split_seed,
        **cluster_metrics,
        **novel_metrics,
        **open_metrics,
        "queries": len(query_image_ids),
        "gallery_images": len(db_image_ids),
        "query_known_images": int(known_mask.sum()),
        "query_novel_images": int(novel_mask.sum()),
        "novel_identities": len(novel_identities),
        "predicted_clusters": cluster_metrics["n_pred_clusters"],
        "config_source": config.get("config_source"),
        "cw_threshold": config["cw_threshold"],
        "cw_top_k": config["cw_top_k"],
        "tau_attach": config["tau_attach"],
        "tau_margin": config["tau_margin"],
    }
    details = pd.DataFrame({
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "dataset": dataset_name,
        "protocol": protocol,
        "split_seed": split_seed,
        "image_id": query_image_ids,
        "open_role": query_open_role,
        "true_identity": y_query,
        "raw_cluster_id": cluster_ids,
        "predicted_label": predicted,
        "best_known_identity": best_names.astype(str),
        "best_known_score": best_scores,
        "second_best_score": second_scores,
        "known_margin": margins,
    })
    print(
        f"{dataset_name} seed={split_seed}: ARI={result['ARI']:.4f} AMI={result['AMI']:.4f} "
        f"BCubedF1={result['BCubed_F1']:.4f} novel_ARI={result['novel_only_ARI']:.4f} "
        f"BAKS={result['BAKS']:.4f} BAUS={result['BAUS']:.4f} NormAcc={result['NormAcc']:.4f} "
        f"clusters={result['predicted_clusters']}"
    )
    return result, details


result_rows = []
detail_frames = []
for animal in loaded_animals:
    result, details = evaluate_animal(animal)
    if animal["protocol"] == "query_only_defaults" and (
        int(result["gallery_images"]) != 0
        or int(result["queries"]) != len(animal["query_image_ids"])
    ):
        raise RuntimeError(f"{animal['dataset']}: Jaguar protocol did not use every query image with zero gallery")
    result_rows.append(result)
    detail_frames.append(details)

results = pd.DataFrame(result_rows)
detail_columns = [
    "global_descriptor", "dataset", "protocol", "split_seed", "image_id", "open_role",
    "true_identity", "raw_cluster_id", "predicted_label", "best_known_identity",
    "best_known_score", "second_best_score", "known_margin",
]
aligned = []
for frame in detail_frames:
    piece = frame.copy()
    for column in detail_columns:
        if column not in piece.columns:
            if column in {"best_known_score", "second_best_score", "known_margin"}:
                piece[column] = np.nan
            else:
                piece[column] = ""
    aligned.append(piece[detail_columns])
details = pd.concat(aligned, ignore_index=True)
if set(results["dataset"]) != {job["dataset"] for job in jobs}:
    raise RuntimeError("03c did not produce one result row per selected animal")

finite_metric_cols = [
    "ARI", "AMI", "NMI", "V_measure", "homogeneity", "completeness",
    "BCubed_precision", "BCubed_recall", "BCubed_F1",
    "pairwise_precision", "pairwise_recall", "pairwise_F1", "BAUS",
]
for column in finite_metric_cols:
    if column not in results.columns:
        raise RuntimeError(f"missing metric column {column}")
    if not np.isfinite(results[column]).all():
        raise RuntimeError(f"03c produced non-finite {column}")
gallery_rows = results.loc[results["protocol"].eq("gallery_attach_optuna")]
if len(gallery_rows):
    for column in ("BAKS", "NormAcc", "known_attachment_accuracy"):
        if not np.isfinite(gallery_rows[column]).all():
            raise RuntimeError(f"gallery animals must have finite {column}")
jaguar_rows = results.loc[results["protocol"].eq("query_only_defaults")]
if len(jaguar_rows) and jaguar_rows["BAKS"].notna().any():
    raise RuntimeError("query-only animals must leave BAKS as NaN")
print("Finite cluster + BAUS metrics OK. Jaguar BAKS NaN is expected (no known queries).")

results_path = OUTPUT_ROOT / f"discovery_ari_results_{GLOBAL_DESCRIPTOR}.csv"
details_path = OUTPUT_ROOT / f"discovery_predictions_{GLOBAL_DESCRIPTOR}.csv"
results.to_csv(results_path, index=False)
macro_row = {"global_descriptor": GLOBAL_DESCRIPTOR, "all_query_datasets": len(results), "gallery_datasets": len(gallery_rows)}
macro_row.update(results[["ARI", "BCubed_F1", "pairwise_F1"]].mean().to_dict())
macro_row.update(gallery_rows[["novel_only_ARI", "BAKS", "NormAcc"]].mean().to_dict())
macro_path = OUTPUT_ROOT / f"paper_metric_macro_{GLOBAL_DESCRIPTOR}.csv"
pd.DataFrame([macro_row]).to_csv(macro_path, index=False)
details.to_csv(details_path, index=False)
for dataset_name, group in results.groupby("dataset"):
    group.to_csv(OUTPUT_ROOT / f"discovery_ari_results_{GLOBAL_DESCRIPTOR}_{dataset_name}.csv", index=False)
summary_cols = [
    "dataset", "protocol", "ARI", "AMI", "NMI", "V_measure", "homogeneity", "completeness",
    "BCubed_F1", "pairwise_F1", "novel_only_ARI", "novel_only_BCubed_F1",
    "BAKS", "BAUS", "NormAcc", "known_attachment_accuracy",
    "known_to_new_rate", "novel_false_attach_rate",
    "n_true_identities", "predicted_clusters",
]
print(results[summary_cols].to_string(index=False))
print("Mean ARI:", float(results["ARI"].mean()))
print("Mean AMI:", float(results["AMI"].mean()))
print("Mean BCubed F1:", float(results["BCubed_F1"].mean()))
print("Mean BAUS:", float(results["BAUS"].mean()))
if gallery_rows is not None and len(gallery_rows):
    print("Mean gallery BAKS:", float(gallery_rows["BAKS"].mean()))
    print("Mean gallery NormAcc:", float(gallery_rows["NormAcc"].mean()))
print("Full metrics CSV:", results_path)

In [ ]:
# Reproducibility manifest and completion checks.
manifest = {
    "stage": "03c_frozen_open_set_clustering",
    "status": "passed",
    "datasets": list(results["dataset"]),
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "per_animal_thresholds": True,
    "animalclef_repository": "https://github.com/MIA-AI-Team/AnimalCLEF26.git",
    "animalclef_commit": "inlined_from_AnimalCLEF26_cd3d015",
    "threshold_fitting_in_stage03c": False,
    "feature_extraction_in_stage03c": False,
    "animals": {
        animal["dataset"]: {
            "protocol": animal["protocol"],
            "source_matrix": str(animal["matrix_path"]),
            "source_matrix_sha256": hashlib.sha256(animal["matrix_path"].read_bytes()).hexdigest(),
            "source_config": str(animal["config_path"]),
            "source_config_sha256": hashlib.sha256(animal["config_path"].read_bytes()).hexdigest(),
            "config_source": animal["config"].get("config_source"),
            "cw_threshold": animal["config"]["cw_threshold"],
            "cw_top_k": animal["config"]["cw_top_k"],
            "use_rerank": animal["config"]["use_rerank"],
        }
        for animal in loaded_animals
    },
    "mean_ari": float(results["ARI"].mean()),
    "std_ari": float(results["ARI"].std(ddof=1)) if len(results) > 1 else 0.0,
    "mean_novel_only_ari": float(gallery_rows["novel_only_ARI"].mean()) if len(gallery_rows) else None,
    "mean_known_attachment_accuracy": (
        None if results["known_attachment_accuracy"].isna().all()
        else float(results.loc[results["known_attachment_accuracy"].notna(), "known_attachment_accuracy"].mean())
    ),
    "results_file": results_path.name,
    "paper_macro_file": macro_path.name,
    "predictions_file": details_path.name,
}
(OUTPUT_ROOT / "stage03c_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(manifest, indent=2, sort_keys=True))
print(f"\nCOMPLETE: save {OUTPUT_ROOT} as the {GLOBAL_DESCRIPTOR} ARI table across {list(results['dataset'])}.")